# 01 — Exploratory Data Analysis

**Customer Complaint Analyzer**

This notebook explores the filtered CFPB Consumer Complaint dataset to understand:
- Class distribution and imbalance
- Text length characteristics
- Most frequent words per product category
- Issue group distribution
- Temporal trends

Run `python src/data_prep.py` before this notebook.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud
from collections import Counter
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
matplotlib.rcParams['figure.dpi'] = 120
pd.set_option('display.max_colwidth', 100)

BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'data' / 'processed'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

print('Libraries loaded.')

## 1. Load Dataset

In [ ]:
df = pd.read_csv(DATA_DIR / 'full_filtered.csv', parse_dates=['Date received'])

print(f'Shape: {df.shape}')
print(f'Date range: {df["Date received"].min().date()} → {df["Date received"].max().date()}')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# Dataset summary
print('=== Dataset Summary ===')
print(f'Total complaints:     {len(df):,}')
print(f'Unique products:      {df["Product"].nunique()}')
print(f'Unique issue groups:  {df["issue_group"].nunique()}')
print(f'Priority distribution:')
print(df['priority'].value_counts().to_string())
print()
print('Missing values:')
print(df.isnull().sum())

## 2. Product (Label) Distribution

In [ ]:
product_counts = df['Product'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(product_counts.index[::-1], product_counts.values[::-1],
               color=sns.color_palette('muted', len(product_counts)))

for bar, val in zip(bars, product_counts.values[::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

ax.set_xlabel('Number of Complaints', fontsize=12)
ax.set_title('Product Category Distribution (Raw Filtered Data)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'eda_product_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClass proportions:')
print((product_counts / len(df) * 100).round(2).to_string())

In [ ]:
# Load balanced train data for post-balancing distribution
train_df = pd.read_csv(DATA_DIR / 'train.csv')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

raw_counts = df['Product'].value_counts()
balanced_counts = train_df['Product'].value_counts()

for ax, counts, title in zip(axes,
                              [raw_counts, balanced_counts],
                              ['Before Balancing (Full Filtered)', 'After Balancing (Train Split)']):
    ax.barh(counts.index[::-1], counts.values[::-1],
            color=sns.color_palette('muted', len(counts)))
    ax.set_xlabel('Count')
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Class Distribution: Before vs After Balancing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'eda_balance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Text Length Distribution

In [ ]:
df['word_count'] = df['Consumer complaint narrative'].str.split().str.len()

print('Word count statistics:')
print(df['word_count'].describe().round(1))
print(f'\nPercentiles:')
for p in [50, 75, 90, 95, 99]:
    print(f'  {p}th percentile: {df["word_count"].quantile(p/100):.0f} words')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall histogram
axes[0].hist(df['word_count'].clip(upper=500), bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(128, color='red', linestyle='--', linewidth=1.5, label='128 tokens')
axes[0].axvline(256, color='orange', linestyle='--', linewidth=1.5, label='256 tokens')
axes[0].set_xlabel('Word Count (clipped at 500)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Narrative Length Distribution', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Box plot by product
plot_data = df[df['word_count'] < 500]
product_order = plot_data.groupby('Product')['word_count'].median().sort_values().index
axes[1].boxplot(
    [plot_data[plot_data['Product'] == p]['word_count'].values for p in product_order],
    labels=[p[:20] for p in product_order],
    vert=False, patch_artist=True,
    boxprops=dict(facecolor='lightblue', color='steelblue'),
    medianprops=dict(color='darkred', linewidth=2),
    showfliers=False,
)
axes[1].set_xlabel('Word Count', fontsize=11)
axes[1].set_title('Narrative Length by Product', fontsize=13, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'eda_text_length.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Most Frequent Words per Category (Word Clouds)

In [ ]:
import re

STOPWORDS = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're",
    "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he',
    'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's",
    'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which',
    'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are',
    'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does',
    'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as',
    'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between',
    'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from',
    'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further',
    'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'both',
    'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not',
    'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will',
    'just', 'don', "don't", 'should', "should've", 'now', 'd', 'll', 'm', 'o', 're',
    've', 'y', 'ain', 'aren', "aren't", 'couldn', "couldn't", 'didn', "didn't",
    'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't",
    'isn', "isn't", 'ma', 'mightn', "mightn't", 'mustn', "mustn't", 'needn',
    "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren',
    "weren't", 'won', "won't", 'wouldn', "wouldn't", 'redacted', 'xxxx', 'xx',
    'said', 'told', 'called', 'would', 'also', 'get', 'got', 'one', 'two', 'three',
    'could', 'make', 'made', 'like', 'know', 'still', 'even', 'back', 'never',
    'however', 'well', 'since', 'yet', 'without', 'upon', 'within', 'us', 'per',
}

def get_top_words(text_series, n=30):
    all_words = []
    for text in text_series.dropna():
        words = re.findall(r'\b[a-z]{3,}\b', text.lower())
        all_words.extend([w for w in words if w not in STOPWORDS])
    return Counter(all_words).most_common(n)

# Test on first product
top_products = df['Product'].value_counts().head(6).index.tolist()
print('Products for word clouds:', top_products)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, product in zip(axes, top_products):
    subset = df[df['Product'] == product]['Consumer complaint narrative']
    top_words = get_top_words(subset, n=100)
    freq_dict = dict(top_words)

    wc = WordCloud(
        width=500, height=300,
        background_color='white',
        colormap='viridis',
        max_words=60,
        prefer_horizontal=0.8,
    ).generate_from_frequencies(freq_dict)

    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(product[:35], fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Most Frequent Words per Product Category', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'eda_word_clouds.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Issue Group Distribution

In [ ]:
issue_counts = df['issue_group'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
sns.barplot(y=issue_counts.index, x=issue_counts.values, ax=axes[0],
            palette='Set2', orient='h')
axes[0].set_xlabel('Count', fontsize=11)
axes[0].set_title('Issue Group Distribution', fontsize=13, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Pie chart
axes[1].pie(issue_counts.values, labels=issue_counts.index,
            autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('Set2', len(issue_counts)))
axes[1].set_title('Issue Group Share', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'eda_issue_groups.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCoverage of issue_group mapping:')
unmapped = (df['issue_group'] == 'Other').sum()
print(f'  Mapped:   {len(df) - unmapped:,} ({(1 - unmapped/len(df))*100:.1f}%)')
print(f'  Unmapped: {unmapped:,} ({unmapped/len(df)*100:.1f}%)')

In [ ]:
# What issues are landing in 'Other'? (useful for refining the map)
unmapped_issues = df[df['issue_group'] == 'Other']['Issue'].value_counts().head(20)
print('Top unmapped issues (falling into "Other"):')
print(unmapped_issues.to_string())

## 6. Temporal Trends

In [ ]:
df['year_month'] = df['Date received'].dt.to_period('M').astype(str)

monthly = df.groupby(['year_month', 'Product']).size().reset_index(name='count')

fig = px.line(
    monthly, x='year_month', y='count', color='Product',
    title='Monthly Complaint Volume by Product (Aug 2023 onward)',
    labels={'year_month': 'Month', 'count': 'Complaints'},
    height=450,
)
fig.update_layout(legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.write_html(str(RESULTS_DIR / 'eda_temporal_trends.html'))
fig.show()

## 7. Sample Complaints per Class

In [ ]:
print('=== Sample Complaints per Product ===\n')
for product in df['Product'].unique():
    sample = df[df['Product'] == product]['Consumer complaint narrative'].iloc[0]
    print(f'[{product}]')
    print(' '.join(sample.split()[:60]) + ' ...')
    print()

## 8. Sequence Length Analysis — Token Cutoff Decision

In [ ]:
# How much content is captured at 128 vs 256 tokens?
wc = df['word_count']

for cutoff in [64, 128, 192, 256, 512]:
    pct = (wc <= cutoff).mean() * 100
    print(f'max_length={cutoff:>4}: captures {pct:.1f}% of complaints fully')

print(f'\nMedian word count: {wc.median():.0f}')
print(f'Mean word count:   {wc.mean():.0f}')

## 9. Priority Distribution

In [ ]:
priority_order = ['Critical', 'High', 'Medium', 'Low']
priority_counts = df['priority'].value_counts().reindex(priority_order, fill_value=0)

colors = ['#d62728', '#ff7f0e', '#ffdd57', '#2ca02c']
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(priority_counts.index, priority_counts.values, color=colors)

for bar, val in zip(bars, priority_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)

ax.set_title('Complaint Priority Distribution (Rule-Based)', fontsize=13, fontweight='bold')
ax.set_ylabel('Count', fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'eda_priority_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Train / Val / Test Split Verification

In [ ]:
val_df = pd.read_csv(DATA_DIR / 'val.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'Total: {len(train_df) + len(val_df) + len(test_df):,}')

# Verify stratification
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, split_df, name in zip(axes,
                               [train_df, val_df, test_df],
                               ['Train (70%)', 'Val (15%)', 'Test (15%)']):
    counts = split_df['Product'].value_counts(normalize=True).sort_index()
    ax.barh(counts.index, counts.values * 100,
            color=sns.color_palette('muted', len(counts)))
    ax.set_xlabel('Percentage (%)')
    ax.set_title(name, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Class Distribution Across Splits (Stratified)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'eda_split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Key findings from EDA:
1. **Class imbalance** — Credit Reporting dominates; balancing was necessary.
2. **Text length** — Median ~80 words; 95th percentile ~250. Both 128 and 256 token cutoffs are reasonable.
3. **Issue mapping** — Review the 'Other' bucket above; if >20% unmapped, add more mappings.
4. **Priority** — Majority of complaints are Low/Medium priority; High/Critical are minority.
5. **Temporal** — Check for trend shifts that might affect generalization.

**Decision on max_length:** See cell 8 output — use 256 if >15% of complaints exceed 128 words; otherwise 128 is sufficient and trains faster.